This tutorial is going to follow [Pr. Gaceb first Deep Learning lab](../labs/DL/TP1.ipynb)'s structure in addition to following the PyTorch official documentation (referenced in each relevant section)

# Exo 2 & 3 - Tensors (torch.tensor) and NumPy

In [18]:
import torch
import numpy as np

Les tenseurs PyTorch (torch.tensor) sont des structures de données multidimensionnelles fondamentales, similaires aux tableaux NumPy, mais optimisées pour le calcul sur GPU et la différenciation automatique dans le deep learning.

Ils stockent des données (entrées/sorties) et les paramètres de modèle (poids/gradients), permettant des opérations rapides.

Une valeur scalaire est représentée par un tenseur à 0 dimension. De même, un tableau colonne/ligne est représenté par un tenseur 1D, et ainsi de suite...

<div align="center">
    <img src="../../assets/imgs/tensor_illustration.png" width=400>
</div>

Les tensors peuvent être créés à partir de listes Python, de tableaux NumPy, ou initialisés avec des valeurs spécifiques (zéros, uns, valeurs aléatoires).

The API is similar to [Numpy](NumPy.ipynb)'s and PyTorch is NumPy-compatible:

In [26]:
t = torch.tensor([
    [ [1.0, 2.0],
      [3.0, 4.0]
    ],
    [ [5.0, 6.0],
      [7.0, 8.0]
    ]
])
print(f"\n{t=}")
print(f"\n{t.numpy()=}")
print(f"\n{torch.from_numpy(t.numpy())=}")


t=tensor([[[1., 2.],
         [3., 4.]],

        [[5., 6.],
         [7., 8.]]])

t.numpy()=array([[[1., 2.],
        [3., 4.]],

       [[5., 6.],
        [7., 8.]]], dtype=float32)

torch.from_numpy(t.numpy())=tensor([[[1., 2.],
         [3., 4.]],

        [[5., 6.],
         [7., 8.]]])


# Exo 4 - Tensor Operations

## 4.1. Addition
L'opération la plus simple consiste à additionner deux tenseurs.

In [29]:
x1 = torch.rand(3, 2)
x2 = torch.rand(3, 2)
print(f"{x1=}")
print(f"{x2=}")

y = x1 + x2
print(f"{y=}")

x1=tensor([[0.4899, 0.4069],
        [0.9482, 0.5039],
        [0.1234, 0.5116]])
x2=tensor([[0.0620, 0.2155],
        [0.3507, 0.1245],
        [0.0526, 0.4731]])
y=tensor([[0.5519, 0.6224],
        [1.2989, 0.6285],
        [0.1760, 0.9847]])


To better illustrate the operation:

In [30]:
from mlkit.util.prettyprint import tensor_add
tensor_add(x1, x2)

[
  [0.4899 + 0.0620, 0.4069 + 0.2155],
  [0.9482 + 0.3507, 0.5039 + 0.1245],
  [0.1234 + 0.0526, 0.5116 + 0.4731]
]


L'appel à `x1 + x2` crée un nouveau tenseur contenant la somme des deux entrées. Cependant, il est également possible d'utiliser des opérations sur place, appliquées directement à la mémoire d'un tenseur. On modifie ainsi les valeurs de `x2` sans avoir à y accéder à nouveau avant l'opération.

In [32]:
x2.add_(x1)
print(f"{x2=}")

x2=tensor([[1.0418, 1.0293],
        [2.2471, 1.1324],
        [0.2994, 1.4963]])


Les opérations sur `n` éléments sont généralement précédées d'un tiret bas (par exemple, `add_` au lieu de `add`).

## 4.2. Gradients

This is not mentioned in Pr. Gaceb's lab but I thought it was important to highlight it while attempting Exo 6. The example below comes from [this seciton](https://docs.pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html#differentiation-in-autograd) of the official PyTorch beginner's documentation.

`torch.autograd` is PyTorch’s automatic differentiation engine that works by collecting gradients after each operation on a tracked tensor. Let's look at an example.

We create two tensors `a` and `b` with `requires_grad=True`. This signals to `autograd` that every operation on them should be tracked.

In [47]:
a = torch.tensor([2., 3.], requires_grad=True)
b = torch.tensor([6., 4.], requires_grad=True)

We create another tensor `Q` from `a` and `b`

$$ Q = 3a^3 - b^2 $$

More precisely,
$$ Q = \begin{pmatrix}       Q_1      &      Q_2       \\ \end{pmatrix}
     = \begin{pmatrix} 3(a_1)^3 - (b_1)^2 & 3(a_2)^3 - (b_2)^2 \\ \end{pmatrix} $$

and so pluging in the values of `a` and `b` we get:

$$ Q = \begin{pmatrix} 3(a_1)^3 - (b_1)^2 & 3(a_2)^3 - (b_2)^2 \\ \end{pmatrix}
     = \begin{pmatrix} 3\times(2.0)^3 - (6.0)^2 & 3\times(3.0)^3 - (4.0)^2 \\ \end{pmatrix}
     = \begin{pmatrix} -12.0 & 65.0 \end{pmatrix} $$

In [49]:
Q = 3*a**3 - b**2
print(f"{Q=}")

Q=tensor([-12.,  65.], grad_fn=<SubBackward0>)


That `SubBackward0` is the last operation that the tracked tensors underwent. See the official documentation for more information on the [computational graph](https://docs.pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html#computational-graph)

Let’s assume `a` and `b` to be parameters of an NN, and `Q` to be the error. In NN training, we want gradients of the error w.r.t. parameters, i.e. $\frac{\partial{Q}}{\partial{a}}$ and $\frac{\partial{Q}}{\partial{b}}$. Notice however, that $Q$ is a function of multiple variables $Q(a, b)$, variables which are vectors themselves, with multiple components. So the derivative will have include each output $(Q_1, Q_2)$ w.r.t. each input $(a_1, a_2)$ and $(b_1, b_2)$. Therefore

$$ \frac{\partial{Q}}{\partial{a}} = \begin{pmatrix}
                                        \frac{\partial{Q_1}}{\partial{a_1}} & \frac{\partial{Q_1}}{\partial{a_2}} \\
                                        \frac{\partial{Q_2}}{\partial{a_1}} & \frac{\partial{Q_2}}{\partial{a_2}} \\
                                     \end{pmatrix}
                                   = \begin{pmatrix}
                                                  9(a_1)^2                    &                 0                   \\
                                                    0                         &               9(a_2)^2              \\
                                     \end{pmatrix} $$
                                   
$$ \frac{\partial{Q}}{\partial{b}} = \begin{pmatrix}
                                        \frac{\partial{Q_1}}{\partial{b_1}} & \frac{\partial{Q_1}}{\partial{b_2}} \\
                                        \frac{\partial{Q_2}}{\partial{b_1}} & \frac{\partial{Q_2}}{\partial{b_2}} \\
                                     \end{pmatrix}
                                   = \begin{pmatrix}
                                                  -2b_1                     &                 0                   \\
                                                    0                       &                -2b_2                \\
                                     \end{pmatrix} $$

**Note**: those derivation matrices are refered to as [Jacobian](https://en.wikipedia.org/wiki/Jacobian_matrix_and_determinant) matrices: the generalization of single variable derivation to multiple variable vectors

and so pluging in the values of `a` and `b` we get (on the diagonals):

$$ \frac{\partial{Q}}{\partial{a}} = \begin{pmatrix} 9(a_1)^2 & 9(a_2)^2 \\ \end{pmatrix} = \begin{pmatrix} 9\times(2.0)^2 & 9\times(3.0)^2 \\ \end{pmatrix} = \begin{pmatrix} 36 & 81 \end{pmatrix} $$
$$ \frac{\partial{Q}}{\partial{b}} = \begin{pmatrix} -2b_1 & -2b_2 \\ \end{pmatrix} = \begin{pmatrix} -2 \times 6 & -2 \times 4 \\ \end{pmatrix} = \begin{pmatrix} -12 & -8 \\ \end{pmatrix} $$

Those same results can be obtained using PyTorch's `.backward()` function which computes something called the **vector-Jacobian product**: $v^TJ$ because `.backward()` doesn't return a full Jacobian, it can only return and propagate one vector backward through the graph, so it needs to be told which one. Because of $Q$'s structure, we'll take the column sum by passing `v=[1., 1.]`:

$$ v^T J = \begin{pmatrix}1.0 \space 1.0\end{pmatrix}^T J $$

with the Jacobian $J$ being the two derivations $\frac{\partial{Q}}{\partial{a}}$ and $\frac{\partial{Q}}{\partial{b}}$:

$$ v^T J = v^T \frac{\partial{Q}}{\partial{a}}
         = \begin{pmatrix}1.0 \space 1.0\end{pmatrix}^T
           \begin{pmatrix}
              36.0 &  0   \\
               0   & 81.0 \\
           \end{pmatrix}
         = \begin{pmatrix} 36 & 81 \end{pmatrix} $$
         
$$ v^T J = v^T \frac{\partial{Q}}{\partial{b}}
         = \begin{pmatrix}1.0 \space 1.0\end{pmatrix}^T
           \begin{pmatrix}
             -12.0 &  0   \\
               0   & -8.0 \\
           \end{pmatrix}
         = \begin{pmatrix} -12.0 & -8.0 \end{pmatrix} $$

In [ ]:
v=torch.tensor([1., 1.])
Q.backward(gradient=v)

In [52]:
print(f"{a.grad=}")
print(f"{b.grad=}")

a.grad=tensor([36., 81.])
b.grad=tensor([-12.,  -8.])


## 4.3. Sur les Matrices
Parmi les autres opérations couramment utilisées figurent les multiplications matricielles essentielles pour les réseaux de neurones. On dispose souvent d'un vecteur d'entrée transformé à l'aide d'une matrice de poids apprise. Il existe plusieurs méthodes et fonctions pour effectuer une multiplication matricielle, dont certaines sont listées ci-dessous :
- `torch.matmul` : Effectue le produit matriciel de deux tenseurs. Son comportement précis dépend de leurs dimensions. Si les deux entrées sont des matrices (tenseurs bidimensionnels), elle effectue le produit matriciel standard. Pour les entrées de dimension supérieure, la fonction prend en charge le broadcasting (voir la documentation sur le site de Pytorch pour plus de détails). Elle peut également s'écrire sous la forme `a @ b`, comme avec NumPy.
- `torch.mm` : Effectue le produit matriciel de deux matrices, mais ne prend pas en charge le broadcasting (voir la documentation).
- `torch.bmm` : Effectue le produit matriciel en prenant en charge la dimension de lot. Si le premier tenseur $T$ est de forme $b \times n \times m$ et le second $R$ de forme $B \times m \times p$, la sortie $O$ est de forme $b \times n \times p$ et a été calculée en effectuant $b$ multiplications matricielles des sous-matrices de $T$ et $R$: $O_i = T_i @ R_i$
- `torch.einsum` : Effectue des multiplications matricielles et d'autres opérations (par exemple, des sommes de produits) en utilisant la convention de sommation d'Einstein.

In [19]:
W = torch.arange(9).view(3, 3) # We can also stack multiple operations in a single line
print("W", W)
h = torch.matmul(x, W) # Verify the result by calculating it by hand too!
print("h", h)

W tensor([[0, 1, 2],
        [3, 4, 5],
        [6, 7, 8]])
h tensor([[15, 18, 21],
        [42, 54, 66]])
